In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install streamlit -q
!npm install -g localtunnel

⠙⠹⠸⠼
changed 22 packages in 571ms
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [ ]:
%%writefile app.py
import numpy as np
import streamlit as st
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib

from pathlib import Path
from PIL import Image


# ==========================================================
# Configuración general
# ==========================================================

st.set_page_config(
    page_title="Clasificador de radiografías de tórax",
    layout="wide"
)


# ==========================================================
# Rutas de modelos
# ==========================================================

RUTA_MODELO_CNN_3_CLASES = Path(
    "/content/drive/MyDrive/TFG/modelos/mejor_modelo_radiografias.keras"
)

RUTA_MODELO_CNN_4_CLASES = Path(
    "/content/drive/MyDrive/TFG/modelos con 2 datasets/mejor_modelo_radiografias_2_datasets.keras"
)

RUTA_MODELO_RESNET50 = Path(
    "/content/drive/MyDrive/TFG/modelos resnet50 preentrenada 1 dataset/mejor_modelo_resnet50_preentrenada_1dataset.keras"
)

RUTA_MODELO_RF = Path(
    "/content/drive/MyDrive/TFG/modelos random forest/random_forest_3_clases.joblib"
)

RUTA_PCA_RF = Path(
    "/content/drive/MyDrive/TFG/modelos random forest/pca_random_forest_3_clases.joblib"
)


RUTA_EXTRACTOR_HIBRIDO_CNN = Path(
    "/content/drive/MyDrive/TFG/modelos hibrido cnn propia reentrenada random forest/extractor_cnn_propia.keras"
)

RUTA_MODELO_RF_HIBRIDO_CNN = Path(
    "/content/drive/MyDrive/TFG/modelos hibrido cnn propia reentrenada random forest/random_forest_hibrido_gridsearch.joblib"
)

RUTA_ENCODER_AUTOENCODER = Path(
    "/content/drive/MyDrive/TFG/modelos autoencoder random forest/encoder_convolucional.keras"
)

RUTA_MODELO_RF_AUTOENCODER = Path(
    "/content/drive/MyDrive/TFG/modelos autoencoder random forest/random_forest_autoencoder_gridsearch.joblib"
)


# ==========================================================
# Clases
# ==========================================================

CLASES_CNN_3 = [
    "COVID",
    "Normal",
    "Viral Pneumonia"
]

CLASES_CNN_4 = [
    "COVID",
    "Normal",
    "Viral Pneumonia",
    "Bacterial Pneumonia"
]

CLASES_RESNET50 = [
    "COVID",
    "Normal",
    "Viral Pneumonia"
]

CLASES_RF = [
    "COVID",
    "Normal",
    "Viral Pneumonia"
]


CLASES_HIBRIDO_CNN_RF = [
    "COVID",
    "Normal",
    "Viral Pneumonia"
]

CLASES_AUTOENCODER_RF = [
    "COVID",
    "Normal",
    "Viral Pneumonia"
]


# ==========================================================
# Tamaños de entrada
# ==========================================================

TAMANO_IMAGEN_CNN = (256, 256)
TAMANO_IMAGEN_RESNET50 = (224, 224)
TAMANO_IMAGEN_RF = (64, 64)
TAMANO_IMAGEN_AUTOENCODER = (128, 128)


# ==========================================================
# Capas personalizadas
# ==========================================================

@tf.keras.utils.register_keras_serializable(package="TFG")
class NormalizacionPorImagen(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, entradas):
        entradas = tf.cast(entradas, tf.float32)

        media = tf.reduce_mean(
            entradas,
            axis=[1, 2, 3],
            keepdims=True
        )

        desviacion = tf.math.reduce_std(
            entradas,
            axis=[1, 2, 3],
            keepdims=True
        )

        numero_pixeles = tf.cast(
            tf.reduce_prod(tf.shape(entradas)[1:]),
            tf.float32
        )

        desviacion_minima = 1.0 / tf.sqrt(numero_pixeles)

        desviacion_ajustada = tf.maximum(
            desviacion,
            desviacion_minima
        )

        return (entradas - media) / desviacion_ajustada

    def get_config(self):
        config = super().get_config()
        return config


@tf.keras.utils.register_keras_serializable(package="TFG")
class PreprocesadoResNet50(tf.keras.layers.Layer):
    """
    Capa usada por el modelo ResNet50 preentrenado.

    Aplica el preprocesado oficial de ResNet50:
    - espera valores de píxel en rango 0-255
    - convierte RGB a BGR
    - centra los canales según ImageNet

    Por eso en la app NO dividimos entre 255 para ResNet50.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, entradas):
        from tensorflow.keras.applications.resnet50 import preprocess_input

        entradas = tf.cast(entradas, tf.float32)

        return preprocess_input(entradas)

    def get_config(self):
        config = super().get_config()
        return config


# ==========================================================
# Carga de modelos
# Se cargan solo cuando se usan.
# ==========================================================

@st.cache_resource
def cargar_cnn_3_clases():
    modelo = tf.keras.models.load_model(
        str(RUTA_MODELO_CNN_3_CLASES),
        custom_objects={
            "NormalizacionPorImagen": NormalizacionPorImagen,
            "PreprocesadoResNet50": PreprocesadoResNet50
        },
        compile=False
    )

    return modelo


@st.cache_resource
def cargar_cnn_4_clases():
    modelo = tf.keras.models.load_model(
        str(RUTA_MODELO_CNN_4_CLASES),
        custom_objects={
            "NormalizacionPorImagen": NormalizacionPorImagen,
            "PreprocesadoResNet50": PreprocesadoResNet50
        },
        compile=False
    )

    return modelo


@st.cache_resource
def cargar_resnet50():
    modelo = tf.keras.models.load_model(
        str(RUTA_MODELO_RESNET50),
        custom_objects={
            "NormalizacionPorImagen": NormalizacionPorImagen,
            "PreprocesadoResNet50": PreprocesadoResNet50
        },
        compile=False
    )

    return modelo


@st.cache_resource
def cargar_random_forest():
    modelo_rf = joblib.load(RUTA_MODELO_RF)
    pca_rf = joblib.load(RUTA_PCA_RF)

    return modelo_rf, pca_rf


@st.cache_resource
def cargar_hibrido_cnn_random_forest():
    extractor = tf.keras.models.load_model(
        str(RUTA_EXTRACTOR_HIBRIDO_CNN),
        custom_objects={
            "NormalizacionPorImagen": NormalizacionPorImagen,
            "PreprocesadoResNet50": PreprocesadoResNet50
        },
        compile=False
    )

    modelo_rf = joblib.load(RUTA_MODELO_RF_HIBRIDO_CNN)

    return extractor, modelo_rf


@st.cache_resource
def cargar_autoencoder_random_forest():
    encoder = tf.keras.models.load_model(
        str(RUTA_ENCODER_AUTOENCODER),
        compile=False
    )

    modelo_rf = joblib.load(RUTA_MODELO_RF_AUTOENCODER)

    return encoder, modelo_rf


# ==========================================================
# Preprocesado de imágenes
# ==========================================================

def preparar_imagen_cnn(archivo):
    """
    Preprocesado para las CNN propias.

    - Escala de grises.
    - 256x256.
    - Shape final: (1, 256, 256, 1).

    No dividimos entre 255 porque los modelos CNN propios ya incluyen
    una capa Rescaling(1.0 / 255.0).
    """

    archivo.seek(0)

    imagen = Image.open(archivo)
    imagen = imagen.convert("L")
    imagen = imagen.resize(TAMANO_IMAGEN_CNN)

    array = np.array(imagen).astype("float32")

    array = np.expand_dims(array, axis=-1)
    array = np.expand_dims(array, axis=0)

    return imagen, array


def preparar_imagen_resnet50(archivo):
    """
    Preprocesado para ResNet50 preentrenada.

    - RGB.
    - 224x224.
    - Shape final: (1, 224, 224, 3).

    No dividimos entre 255 porque el modelo ResNet50 incluye
    la capa PreprocesadoResNet50.
    """

    archivo.seek(0)

    imagen = Image.open(archivo)
    imagen = imagen.convert("RGB")
    imagen = imagen.resize(TAMANO_IMAGEN_RESNET50)

    array = np.array(imagen).astype("float32")

    array = np.expand_dims(array, axis=0)

    return imagen, array


def preparar_imagen_rf(archivo):
    """
    Preprocesado para Random Forest.

    - Escala de grises.
    - 64x64.
    - Normalización 0-1.
    - Vector plano.
    - Shape final: (1, 4096).
    """

    archivo.seek(0)

    imagen = Image.open(archivo)
    imagen = imagen.convert("L")
    imagen = imagen.resize(TAMANO_IMAGEN_RF)

    array = np.array(imagen).astype("float32")
    array = array / 255.0

    vector = array.flatten()
    vector = np.expand_dims(vector, axis=0)

    return imagen, vector


def preparar_imagen_autoencoder(archivo):
    """
    Preprocesado para Autoencoder + Random Forest.

    - Escala de grises.
    - 128x128.
    - Normalización 0-1.
    - Shape final: (1, 128, 128, 1).

    El encoder del autoencoder fue entrenado con imágenes normalizadas
    antes de entrar en la red, por eso aquí sí dividimos entre 255.
    """

    archivo.seek(0)

    imagen = Image.open(archivo)
    imagen = imagen.convert("L")
    imagen = imagen.resize(TAMANO_IMAGEN_AUTOENCODER)

    array = np.array(imagen).astype("float32")
    array = array / 255.0

    array = np.expand_dims(array, axis=-1)
    array = np.expand_dims(array, axis=0)

    return imagen, array


# ==========================================================
# Predicciones
# ==========================================================

def predecir_con_cnn(modelo, archivo, clases, nombre_modelo):
    imagen_mostrada, imagen_modelo = preparar_imagen_cnn(archivo)

    probabilidades = modelo.predict(
        imagen_modelo,
        verbose=0
    )[0]

    indice_prediccion = int(np.argmax(probabilidades))
    clase_predicha = clases[indice_prediccion]
    confianza = float(probabilidades[indice_prediccion])

    return {
        "tipo_modelo": "cnn",
        "nombre_modelo": nombre_modelo,
        "imagen_mostrada": imagen_mostrada,
        "imagen_modelo": imagen_modelo,
        "probabilidades": probabilidades,
        "indice_prediccion": indice_prediccion,
        "clase_predicha": clase_predicha,
        "confianza": confianza,
        "clases": clases
    }


def predecir_con_resnet50(modelo, archivo):
    imagen_mostrada, imagen_modelo = preparar_imagen_resnet50(archivo)

    probabilidades = modelo.predict(
        imagen_modelo,
        verbose=0
    )[0]

    indice_prediccion = int(np.argmax(probabilidades))
    clase_predicha = CLASES_RESNET50[indice_prediccion]
    confianza = float(probabilidades[indice_prediccion])

    return {
        "tipo_modelo": "resnet50",
        "nombre_modelo": "ResNet50 preentrenada - primer dataset - 3 clases",
        "imagen_mostrada": imagen_mostrada,
        "imagen_modelo": imagen_modelo,
        "probabilidades": probabilidades,
        "indice_prediccion": indice_prediccion,
        "clase_predicha": clase_predicha,
        "confianza": confianza,
        "clases": CLASES_RESNET50
    }


def alinear_probabilidades_random_forest(modelo_rf, probabilidades_originales, clases):
    """
    Coloca las probabilidades del Random Forest en el mismo orden
    utilizado por la interfaz.

    Scikit-learn devuelve las probabilidades en el orden de modelo_rf.classes_.
    En estos experimentos las etiquetas esperadas son 0, 1 y 2.
    """

    probabilidades = np.zeros(
        len(clases),
        dtype="float32"
    )

    for clase_modelo, probabilidad in zip(
        modelo_rf.classes_,
        probabilidades_originales
    ):
        indice = int(clase_modelo)

        if indice < 0 or indice >= len(clases):
            raise ValueError(
                f"Etiqueta inesperada en Random Forest: {clase_modelo}"
            )

        probabilidades[indice] = float(probabilidad)

    return probabilidades


def predecir_con_random_forest(modelo_rf, pca_rf, archivo):
    imagen_mostrada, vector = preparar_imagen_rf(archivo)

    vector_pca = pca_rf.transform(vector)

    probabilidades_originales = modelo_rf.predict_proba(vector_pca)[0]

    probabilidades = alinear_probabilidades_random_forest(
        modelo_rf=modelo_rf,
        probabilidades_originales=probabilidades_originales,
        clases=CLASES_RF
    )

    indice_prediccion = int(np.argmax(probabilidades))
    clase_predicha = CLASES_RF[indice_prediccion]
    confianza = float(probabilidades[indice_prediccion])

    return {
        "tipo_modelo": "random_forest",
        "nombre_modelo": "Random Forest + PCA",
        "imagen_mostrada": imagen_mostrada,
        "imagen_modelo": None,
        "probabilidades": probabilidades,
        "indice_prediccion": indice_prediccion,
        "clase_predicha": clase_predicha,
        "confianza": confianza,
        "clases": CLASES_RF
    }


def predecir_con_hibrido_cnn_random_forest(extractor, modelo_rf, archivo):
    """
    Flujo:
    radiografía 256x256 -> extractor de la CNN propia -> vector de 128
    características -> Random Forest -> probabilidades.
    """

    imagen_mostrada, imagen_modelo = preparar_imagen_cnn(archivo)

    caracteristicas = extractor.predict(
        imagen_modelo,
        verbose=0
    )

    caracteristicas = np.asarray(
        caracteristicas,
        dtype="float32"
    )

    caracteristicas = np.nan_to_num(
        caracteristicas,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    probabilidades_originales = modelo_rf.predict_proba(
        caracteristicas
    )[0]

    probabilidades = alinear_probabilidades_random_forest(
        modelo_rf=modelo_rf,
        probabilidades_originales=probabilidades_originales,
        clases=CLASES_HIBRIDO_CNN_RF
    )

    indice_prediccion = int(np.argmax(probabilidades))
    clase_predicha = CLASES_HIBRIDO_CNN_RF[indice_prediccion]
    confianza = float(probabilidades[indice_prediccion])

    return {
        "tipo_modelo": "hibrido_cnn_rf",
        "nombre_modelo": "CNN propia + Random Forest - modelo híbrido",
        "imagen_mostrada": imagen_mostrada,
        "imagen_modelo": None,
        "probabilidades": probabilidades,
        "indice_prediccion": indice_prediccion,
        "clase_predicha": clase_predicha,
        "confianza": confianza,
        "clases": CLASES_HIBRIDO_CNN_RF
    }


def predecir_con_autoencoder_random_forest(encoder, modelo_rf, archivo):
    """
    Flujo:
    radiografía 128x128 normalizada -> encoder -> vector latente de 128
    características -> Random Forest -> probabilidades.
    """

    imagen_mostrada, imagen_modelo = preparar_imagen_autoencoder(
        archivo
    )

    caracteristicas = encoder.predict(
        imagen_modelo,
        verbose=0
    )

    caracteristicas = np.asarray(
        caracteristicas,
        dtype="float32"
    )

    caracteristicas = np.nan_to_num(
        caracteristicas,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    probabilidades_originales = modelo_rf.predict_proba(
        caracteristicas
    )[0]

    probabilidades = alinear_probabilidades_random_forest(
        modelo_rf=modelo_rf,
        probabilidades_originales=probabilidades_originales,
        clases=CLASES_AUTOENCODER_RF
    )

    indice_prediccion = int(np.argmax(probabilidades))
    clase_predicha = CLASES_AUTOENCODER_RF[indice_prediccion]
    confianza = float(probabilidades[indice_prediccion])

    return {
        "tipo_modelo": "autoencoder_rf",
        "nombre_modelo": "Autoencoder convolucional + Random Forest",
        "imagen_mostrada": imagen_mostrada,
        "imagen_modelo": None,
        "probabilidades": probabilidades,
        "indice_prediccion": indice_prediccion,
        "clase_predicha": clase_predicha,
        "confianza": confianza,
        "clases": CLASES_AUTOENCODER_RF
    }


# ==========================================================
# Gráfico de probabilidades
# ==========================================================

def mostrar_grafico_probabilidades(clases, probabilidades, indice_prediccion):
    fig, ax = plt.subplots(figsize=(8, 4))

    posiciones = np.arange(len(clases))

    colores = []

    for indice in range(len(clases)):
        if indice == indice_prediccion:
            colores.append("#2E86C1")
        else:
            colores.append("#BFC9CA")

    ax.barh(
        posiciones,
        probabilidades * 100,
        color=colores
    )

    ax.set_yticks(posiciones)
    ax.set_yticklabels(clases)

    ax.invert_yaxis()

    ax.set_xlabel("Probabilidad (%)")
    ax.set_xlim(0, 100)

    ax.set_title("Probabilidades por clase")

    for i, valor in enumerate(probabilidades * 100):
        ax.text(
            min(valor + 1, 96),
            i,
            f"{valor:.2f}%",
            va="center"
        )

    st.pyplot(fig)


# ==========================================================
# Grad-CAM para CNN propia
# ==========================================================

def encontrar_ultima_capa_convolucional(modelo):
    for capa in reversed(modelo.layers):
        if isinstance(capa, tf.keras.layers.Conv2D):
            return capa.name

    return None


def generar_gradcam_cnn(modelo, imagen_modelo, indice_clase):
    nombre_capa = encontrar_ultima_capa_convolucional(modelo)

    if nombre_capa is None:
        return None

    capa_convolucional = modelo.get_layer(nombre_capa)

    modelo_gradcam = tf.keras.models.Model(
        inputs=modelo.inputs,
        outputs=[
            capa_convolucional.output,
            modelo.output
        ]
    )

    imagen_tensor = tf.convert_to_tensor(imagen_modelo)

    with tf.GradientTape() as tape:
        salida_conv, predicciones = modelo_gradcam(
            imagen_tensor,
            training=False
        )

        perdida = predicciones[:, indice_clase]

    gradientes = tape.gradient(
        perdida,
        salida_conv
    )

    if gradientes is None:
        return None

    pesos = tf.reduce_mean(
        gradientes,
        axis=(0, 1, 2)
    )

    salida_conv = salida_conv[0]

    mapa_calor = tf.reduce_sum(
        salida_conv * pesos,
        axis=-1
    )

    mapa_calor = tf.maximum(mapa_calor, 0)

    maximo = tf.reduce_max(mapa_calor)

    if maximo == 0:
        return None

    mapa_calor = mapa_calor / maximo

    return mapa_calor.numpy()


# ==========================================================
# Grad-CAM para ResNet50 anidada
# ==========================================================

def aplicar_capa(capa, tensor, entrenamiento=False):
    try:
        return capa(tensor, training=entrenamiento)
    except TypeError:
        return capa(tensor)


def encontrar_modelo_resnet_anidado(modelo):
    for capa in modelo.layers:
        if isinstance(capa, tf.keras.Model) and "resnet" in capa.name.lower():
            return capa

    return None


def generar_gradcam_resnet50(modelo, imagen_modelo, indice_clase):
    base_resnet = encontrar_modelo_resnet_anidado(modelo)

    if base_resnet is None:
        return None

    try:
        ultima_conv = base_resnet.get_layer("conv5_block3_out")
    except Exception:
        return None

    modelo_base_gradcam = tf.keras.models.Model(
        inputs=base_resnet.input,
        outputs=[
            ultima_conv.output,
            base_resnet.output
        ]
    )

    indice_base = None

    for indice, capa in enumerate(modelo.layers):
        if capa.name == base_resnet.name:
            indice_base = indice
            break

    if indice_base is None:
        return None

    imagen_tensor = tf.convert_to_tensor(imagen_modelo)

    with tf.GradientTape() as tape:
        x = imagen_tensor

        for capa in modelo.layers[1:indice_base]:
            x = aplicar_capa(
                capa,
                x,
                entrenamiento=False
            )

        salida_conv, salida_base = modelo_base_gradcam(
            x,
            training=False
        )

        x = salida_base

        for capa in modelo.layers[indice_base + 1:]:
            x = aplicar_capa(
                capa,
                x,
                entrenamiento=False
            )

        predicciones = x
        perdida = predicciones[:, indice_clase]

    gradientes = tape.gradient(
        perdida,
        salida_conv
    )

    if gradientes is None:
        return None

    pesos = tf.reduce_mean(
        gradientes,
        axis=(0, 1, 2)
    )

    salida_conv = salida_conv[0]

    mapa_calor = tf.reduce_sum(
        salida_conv * pesos,
        axis=-1
    )

    mapa_calor = tf.maximum(mapa_calor, 0)

    maximo = tf.reduce_max(mapa_calor)

    if maximo == 0:
        return None

    mapa_calor = mapa_calor / maximo

    return mapa_calor.numpy()


def generar_gradcam(modelo, resultado):
    if resultado["tipo_modelo"] == "cnn":
        return generar_gradcam_cnn(
            modelo=modelo,
            imagen_modelo=resultado["imagen_modelo"],
            indice_clase=resultado["indice_prediccion"]
        )

    if resultado["tipo_modelo"] == "resnet50":
        return generar_gradcam_resnet50(
            modelo=modelo,
            imagen_modelo=resultado["imagen_modelo"],
            indice_clase=resultado["indice_prediccion"]
        )

    return None


def mostrar_gradcam(resultado, mapa_calor):
    if mapa_calor is None:
        st.info(
            "No se ha podido generar el mapa Grad-CAM para este modelo."
        )
        return

    st.write(
        "Interpretación del mapa Grad-CAM: las zonas con colores más cálidos "
        "han tenido mayor influencia en la predicción del modelo, mientras que "
        "las zonas con colores más fríos han tenido menor influencia."
    )

    st.markdown(
        """
        **Escala de importancia aproximada:**

        - **Azul oscuro / azul**: baja influencia en la predicción.
        - **Verde**: influencia intermedia.
        - **Amarillo / naranja**: influencia alta.
        - **Rojo**: mayor influencia en la predicción.
        """
    )

    imagen_pil = resultado["imagen_mostrada"]

    if resultado["tipo_modelo"] == "resnet50":
        tamano = TAMANO_IMAGEN_RESNET50
    else:
        tamano = TAMANO_IMAGEN_CNN

    imagen_rgb = imagen_pil.convert("RGB")
    imagen_rgb = imagen_rgb.resize(tamano)

    mapa_calor_img = Image.fromarray(
        np.uint8(255 * mapa_calor)
    )

    mapa_calor_img = mapa_calor_img.resize(tamano)

    mapa_calor_array = np.array(mapa_calor_img)

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.imshow(imagen_rgb, cmap="gray")

    imagen_mapa = ax.imshow(
        mapa_calor_array,
        cmap="jet",
        alpha=0.35,
        vmin=0,
        vmax=255
    )

    ax.axis("off")

    barra_color = fig.colorbar(
        imagen_mapa,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )

    barra_color.set_label(
        "Importancia para la predicción",
        rotation=270,
        labelpad=18
    )

    barra_color.set_ticks([0, 64, 128, 192, 255])

    barra_color.set_ticklabels(
        [
            "Muy baja",
            "Baja",
            "Media",
            "Alta",
            "Muy alta"
        ]
    )

    st.pyplot(fig)

    st.caption(
        "Nota: Grad-CAM no indica una región diagnosticada clínicamente. "
        "Solo muestra qué zonas han influido más en la decisión del modelo."
    )


# ==========================================================
# Explicación textual y avisos
# ==========================================================

def mostrar_explicacion(resultado):
    clases = resultado["clases"]
    probabilidades = resultado["probabilidades"]
    tipo_modelo = resultado["tipo_modelo"]

    st.subheader("Explicación del resultado")

    probabilidades_ordenadas = sorted(
        zip(clases, probabilidades),
        key=lambda x: x[1],
        reverse=True
    )

    primera_clase, primera_prob = probabilidades_ordenadas[0]
    segunda_clase, segunda_prob = probabilidades_ordenadas[1]

    diferencia = primera_prob - segunda_prob

    st.write(
        f"El modelo ha asignado la mayor probabilidad a **{primera_clase}**, "
        f"con un valor aproximado de **{primera_prob * 100:.2f}%**."
    )

    st.write(
        f"La segunda clase más probable ha sido **{segunda_clase}**, "
        f"con **{segunda_prob * 100:.2f}%**."
    )

    if tipo_modelo == "cnn":
        st.write(
            "El modelo utilizado es una red neuronal convolucional propia. "
            "Este tipo de modelo analiza la imagen mediante filtros convolucionales, "
            "que extraen patrones visuales locales y espaciales de la radiografía."
        )

        st.write(
            "El mapa Grad-CAM ayuda a visualizar qué zonas han influido más en la predicción. "
            "Los colores cálidos, como amarillo, naranja y rojo, indican mayor importancia; "
            "los colores fríos, como azul, indican menor importancia."
        )

    elif tipo_modelo == "resnet50":
        st.write(
            "El modelo utilizado es una ResNet50 preentrenada en ImageNet. "
            "En este caso, ResNet50 actúa como extractor de características visuales, "
            "y sobre esa base se añade una cabeza de clasificación adaptada a las clases del proyecto."
        )

        st.write(
            "Este modelo sirve como comparación frente a la CNN propia, ya que permite evaluar "
            "el rendimiento de una arquitectura estándar preentrenada frente a una arquitectura diseñada específicamente."
        )

    elif tipo_modelo == "hibrido_cnn_rf":
        st.write(
            "El modelo utilizado es un híbrido formado por una CNN propia y un Random Forest. "
            "La CNN actúa como extractor de características visuales: transforma la radiografía "
            "en un vector compacto aprendido de forma supervisada. Después, el Random Forest "
            "clasifica ese vector mediante árboles de decisión."
        )

        st.write(
            "Este experimento permite valorar si las capas convolucionales de la CNN propia "
            "aprenden características útiles incluso cuando se sustituye su clasificador final neuronal."
        )

    elif tipo_modelo == "autoencoder_rf":
        st.write(
            "El modelo utilizado combina un autoencoder convolucional y un Random Forest. "
            "El encoder comprime la radiografía en un vector latente aprendido sin utilizar "
            "las etiquetas de clase. Después, el Random Forest clasifica ese vector."
        )

        st.write(
            "Este experimento permite comparar una reducción no lineal aprendida mediante "
            "reconstrucción frente a PCA y frente a las características supervisadas de la CNN propia."
        )

    elif tipo_modelo == "random_forest":
        st.write(
            "El modelo utilizado es un Random Forest con PCA. "
            "La imagen se convierte en un vector de píxeles, se reduce con PCA "
            "y después se clasifica mediante árboles de decisión."
        )

        st.write(
            "A diferencia de una CNN, este modelo no conserva de forma natural "
            "la estructura espacial de la imagen, por lo que no permite una explicación Grad-CAM."
        )

    if diferencia < 0.15:
        st.warning(
            "La diferencia entre las dos clases más probables es pequeña. "
            "Esto indica que el modelo no tiene una separación clara entre ambas opciones."
        )


def mostrar_aviso_neumonias(resultado):
    clases = resultado["clases"]
    probabilidades = resultado["probabilidades"]
    clase_predicha = resultado["clase_predicha"]

    if "Viral Pneumonia" not in clases:
        return

    indice_viral = clases.index("Viral Pneumonia")
    prob_viral = probabilidades[indice_viral]

    existe_bacterial = "Bacterial Pneumonia" in clases

    if existe_bacterial:
        indice_bacterial = clases.index("Bacterial Pneumonia")
        prob_bacterial = probabilidades[indice_bacterial]

        diferencia = abs(prob_viral - prob_bacterial)

        if (
            clase_predicha in ["Viral Pneumonia", "Bacterial Pneumonia"]
            or diferencia < 0.20
        ):
            st.warning(
                "Aviso: las clases Viral Pneumonia y Bacterial Pneumonia pueden solaparse visualmente. "
                "En los experimentos realizados, estas dos clases han mostrado tendencia a confundirse entre sí. "
                "Por tanto, esta predicción debe interpretarse con cautela."
            )

            st.write(
                f"Probabilidad Viral Pneumonia: **{prob_viral * 100:.2f}%**"
            )

            st.write(
                f"Probabilidad Bacterial Pneumonia: **{prob_bacterial * 100:.2f}%**"
            )

    else:
        if clase_predicha == "Viral Pneumonia":
            st.warning(
                "Aviso: el modelo ha predicho Viral Pneumonia. "
                "Esta clase puede presentar similitudes visuales con otros tipos de neumonía. "
                "La predicción debe interpretarse con cautela."
            )


def comprobar_ruta(ruta, nombre):
    if not ruta.exists():
        st.error(f"No se encontró {nombre}: {ruta}")
        st.stop()


# ==========================================================
# Interfaz principal
# ==========================================================

st.title("Clasificador de radiografías de tórax")

st.write(
    "Aplicación de demostración para comparar modelos de clasificación de radiografías de tórax."
)

st.warning(
    "Esta herramienta es una demostración académica. "
    "No debe utilizarse como sistema de diagnóstico médico real."
)


st.sidebar.title("Modelo")

opcion_modelo = st.sidebar.selectbox(
    "Selecciona el modelo",
    [
        "CNN propia - primer dataset - 3 clases",
        "CNN propia - dos datasets - 4 clases",
        "ResNet50 preentrenada - primer dataset - 3 clases",
        "Random Forest + PCA - baseline clásico",
        "CNN propia + Random Forest - híbrido supervisado",
        "Autoencoder convolucional + Random Forest - híbrido no supervisado"
    ]
)

st.sidebar.markdown("---")

if opcion_modelo == "CNN propia - primer dataset - 3 clases":
    st.sidebar.write("**Clases:**")
    st.sidebar.write("- COVID")
    st.sidebar.write("- Normal")
    st.sidebar.write("- Viral Pneumonia")
    st.sidebar.write("**Entrada:** escala de grises 256x256")
    st.sidebar.write("**Modelo:** CNN propia")

elif opcion_modelo == "CNN propia - dos datasets - 4 clases":
    st.sidebar.write("**Clases:**")
    st.sidebar.write("- COVID")
    st.sidebar.write("- Normal")
    st.sidebar.write("- Viral Pneumonia")
    st.sidebar.write("- Bacterial Pneumonia")
    st.sidebar.write("**Entrada:** escala de grises 256x256")
    st.sidebar.write("**Modelo:** CNN propia ampliada")

elif opcion_modelo == "ResNet50 preentrenada - primer dataset - 3 clases":
    st.sidebar.write("**Clases:**")
    st.sidebar.write("- COVID")
    st.sidebar.write("- Normal")
    st.sidebar.write("- Viral Pneumonia")
    st.sidebar.write("**Entrada:** RGB 224x224")
    st.sidebar.write("**Modelo:** ResNet50 preentrenada")

elif opcion_modelo == "Random Forest + PCA - baseline clásico":
    st.sidebar.write("**Clases:**")
    st.sidebar.write("- COVID")
    st.sidebar.write("- Normal")
    st.sidebar.write("- Viral Pneumonia")
    st.sidebar.write("**Entrada:** 64x64 + PCA")
    st.sidebar.write("**Modelo:** Random Forest")

elif opcion_modelo == "CNN propia + Random Forest - híbrido supervisado":
    st.sidebar.write("**Clases:**")
    st.sidebar.write("- COVID")
    st.sidebar.write("- Normal")
    st.sidebar.write("- Viral Pneumonia")
    st.sidebar.write("**Entrada:** escala de grises 256x256")
    st.sidebar.write("**Extractor:** CNN propia reentrenada")
    st.sidebar.write("**Clasificador:** Random Forest")

else:
    st.sidebar.write("**Clases:**")
    st.sidebar.write("- COVID")
    st.sidebar.write("- Normal")
    st.sidebar.write("- Viral Pneumonia")
    st.sidebar.write("**Entrada:** escala de grises 128x128 normalizada")
    st.sidebar.write("**Extractor:** encoder convolucional")
    st.sidebar.write("**Clasificador:** Random Forest")


if st.session_state.get("ultima_opcion_modelo") != opcion_modelo:
    st.session_state.pop("resultado", None)
    st.session_state.pop("modelo_actual", None)
    st.session_state["ultima_opcion_modelo"] = opcion_modelo


archivo = st.file_uploader(
    "Sube una radiografía de tórax",
    type=["png", "jpg", "jpeg"]
)


if archivo is not None:
    columna_1, columna_2 = st.columns([1, 1])

    with columna_1:
        if opcion_modelo == "Random Forest + PCA - baseline clásico":
            imagen_mostrada_previa, _ = preparar_imagen_rf(archivo)
        elif opcion_modelo == "ResNet50 preentrenada - primer dataset - 3 clases":
            imagen_mostrada_previa, _ = preparar_imagen_resnet50(archivo)
        elif opcion_modelo == "Autoencoder convolucional + Random Forest - híbrido no supervisado":
            imagen_mostrada_previa, _ = preparar_imagen_autoencoder(archivo)
        else:
            imagen_mostrada_previa, _ = preparar_imagen_cnn(archivo)

        st.subheader("Imagen cargada")

        st.image(
            imagen_mostrada_previa,
            caption="Imagen procesada para el modelo seleccionado",
            use_container_width=True
        )

    with columna_2:
        st.subheader("Predicción")

        boton_predecir = st.button("Realizar predicción")

        if boton_predecir:
            with st.spinner("Cargando modelo seleccionado y realizando predicción..."):
                modelo = None

                if opcion_modelo == "CNN propia - primer dataset - 3 clases":
                    comprobar_ruta(
                        RUTA_MODELO_CNN_3_CLASES,
                        "el modelo CNN de 3 clases"
                    )

                    modelo = cargar_cnn_3_clases()

                    resultado = predecir_con_cnn(
                        modelo=modelo,
                        archivo=archivo,
                        clases=CLASES_CNN_3,
                        nombre_modelo="CNN propia - primer dataset - 3 clases"
                    )

                elif opcion_modelo == "CNN propia - dos datasets - 4 clases":
                    comprobar_ruta(
                        RUTA_MODELO_CNN_4_CLASES,
                        "el modelo CNN de 4 clases"
                    )

                    modelo = cargar_cnn_4_clases()

                    resultado = predecir_con_cnn(
                        modelo=modelo,
                        archivo=archivo,
                        clases=CLASES_CNN_4,
                        nombre_modelo="CNN propia - dos datasets - 4 clases"
                    )

                elif opcion_modelo == "ResNet50 preentrenada - primer dataset - 3 clases":
                    comprobar_ruta(
                        RUTA_MODELO_RESNET50,
                        "el modelo ResNet50 preentrenado"
                    )

                    modelo = cargar_resnet50()

                    resultado = predecir_con_resnet50(
                        modelo=modelo,
                        archivo=archivo
                    )

                elif opcion_modelo == "Random Forest + PCA - baseline clásico":
                    comprobar_ruta(
                        RUTA_MODELO_RF,
                        "el modelo Random Forest"
                    )

                    comprobar_ruta(
                        RUTA_PCA_RF,
                        "el PCA del Random Forest"
                    )

                    modelo_rf, pca_rf = cargar_random_forest()

                    resultado = predecir_con_random_forest(
                        modelo_rf=modelo_rf,
                        pca_rf=pca_rf,
                        archivo=archivo
                    )

                elif opcion_modelo == "CNN propia + Random Forest - híbrido supervisado":
                    comprobar_ruta(
                        RUTA_EXTRACTOR_HIBRIDO_CNN,
                        "el extractor CNN del modelo híbrido"
                    )

                    comprobar_ruta(
                        RUTA_MODELO_RF_HIBRIDO_CNN,
                        "el Random Forest del modelo híbrido CNN"
                    )

                    extractor, modelo_rf = cargar_hibrido_cnn_random_forest()

                    resultado = predecir_con_hibrido_cnn_random_forest(
                        extractor=extractor,
                        modelo_rf=modelo_rf,
                        archivo=archivo
                    )

                else:
                    comprobar_ruta(
                        RUTA_ENCODER_AUTOENCODER,
                        "el encoder del autoencoder"
                    )

                    comprobar_ruta(
                        RUTA_MODELO_RF_AUTOENCODER,
                        "el Random Forest del autoencoder"
                    )

                    encoder, modelo_rf = cargar_autoencoder_random_forest()

                    resultado = predecir_con_autoencoder_random_forest(
                        encoder=encoder,
                        modelo_rf=modelo_rf,
                        archivo=archivo
                    )

            st.success(
                f"Clase predicha: {resultado['clase_predicha']}"
            )

            st.metric(
                label="Confianza aproximada",
                value=f"{resultado['confianza'] * 100:.2f}%"
            )

            st.write(f"Modelo usado: **{resultado['nombre_modelo']}**")

            mostrar_aviso_neumonias(resultado)

            st.session_state["resultado"] = resultado

            if modelo is not None:
                st.session_state["modelo_actual"] = modelo
            else:
                st.session_state["modelo_actual"] = None


    if "resultado" in st.session_state:
        resultado = st.session_state["resultado"]

        st.markdown("---")

        st.subheader("Probabilidades por clase")

        mostrar_grafico_probabilidades(
            clases=resultado["clases"],
            probabilidades=resultado["probabilidades"],
            indice_prediccion=resultado["indice_prediccion"]
        )

        st.markdown("---")

        mostrar_explicacion(resultado)

        st.markdown("---")

        if resultado["tipo_modelo"] in ["cnn", "resnet50"]:
            st.subheader("Mapa Grad-CAM")

            st.write(
                "El mapa Grad-CAM resalta de forma aproximada las zonas de la radiografía "
                "que más han influido en la predicción del modelo. "
                "Los colores cálidos, como amarillo, naranja y rojo, representan mayor importancia; "
                "los colores fríos, como azul, representan menor importancia."
            )

            modelo_actual = st.session_state.get("modelo_actual", None)

            if modelo_actual is not None:
                mapa_calor = generar_gradcam(
                    modelo=modelo_actual,
                    resultado=resultado
                )

                mostrar_gradcam(
                    resultado=resultado,
                    mapa_calor=mapa_calor
                )

        elif resultado["tipo_modelo"] == "hibrido_cnn_rf":
            st.info(
                "Este modelo utiliza una CNN para extraer características, "
                "pero la predicción final la realiza un Random Forest. "
                "La implementación estándar de Grad-CAM no puede aplicarse "
                "directamente a la salida del clasificador de árboles."
            )

        elif resultado["tipo_modelo"] == "autoencoder_rf":
            st.info(
                "Este modelo utiliza un encoder convolucional para comprimir la imagen "
                "y un Random Forest para clasificar el vector latente. "
                "La implementación estándar de Grad-CAM no puede aplicarse "
                "directamente a la predicción final del Random Forest."
            )

        else:
            st.info(
                "El modelo Random Forest con PCA no permite Grad-CAM, "
                "ya que no trabaja con mapas espaciales convolucionales."
            )

else:
    st.info("Sube una imagen para comenzar.")


Overwriting app.py


In [ ]:
!pkill -f streamlit
!pkill -f lt

!nohup streamlit run app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/logs_streamlit.txt 2>&1 &

In [ ]:
import time

time.sleep(10)

!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 15 Jun 2026 07:23:59 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 1522
last-modified: Mon, 15 Jun 2026 07:19:27 GMT
etag: "475734e659510ae1569e31dabc0f7cfe"
cache-control: no-cache



In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
!cat /content/logs_streamlit.txt



2026-06-15 07:23:51.806 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.253.159:8501

